In [ ]:
import torch
import numpy as np
import pandas as pd
import cv2
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from torch import Tensor
from torch.nn import Sequential, Linear, ReLU, CrossEntropyLoss, Module, Dropout, Flatten, Conv2d, MaxPool2d, AdaptiveAvgPool2d
from torch.utils.data import DataLoader, Dataset
from torch.optim import Adam
from pathlib import Path
import matplotlib.pyplot as plt
import time
import csv

# Este projeto roda apenas na GPU
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU não encontrada. Verifique se o kernel do notebook é o .venv, "
        f"se o PyTorch é a versão CUDA (instalado: {torch.__version__}) "
        "e se o driver da NVIDIA está instalado (rode nvidia-smi no terminal)."
    )
print(f"GPU: {torch.cuda.get_device_name(0)}")

# Confere se este PyTorch tem suporte à placa (ex.: RTX série 50 precisa do PyTorch com CUDA 13)
major, minor = torch.cuda.get_device_capability(0)
if f"sm_{major}{minor}" not in torch.cuda.get_arch_list():
    raise RuntimeError(
        f"O PyTorch instalado ({torch.__version__}) não tem suporte à {torch.cuda.get_device_name(0)} (sm_{major}{minor}). "
        "Para placas RTX série 50, instale a versão com CUDA 13: "
        ".venv\\Scripts\\python -m pip install --force-reinstall --no-deps "
        "torch==2.14.0+cu130 torchvision==0.29.0+cu130 --index-url https://download.pytorch.org/whl/cu130"
    )

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
class MyDataset(Dataset):
    def __init__(self, datas: np.ndarray, labels: np.ndarray) -> None:
        self._x = (torch.from_numpy(datas).float() / escala).to(device)
        self._y = torch.from_numpy(labels).long().to(device)

    def __len__(self) -> int:
        return len(self._x)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        return self._x[idx], self._y[idx]

In [ ]:
class CustomModel(Module):
    def __init__(self, input_size: int, output_size: int) -> None:
        super().__init__()

        self.net = Sequential(
            Conv2d(3, 16, kernel_size=3, padding=1),
            ReLU(),
            MaxPool2d(kernel_size=2),
            Conv2d(16, 32, kernel_size=3, padding=1),
            ReLU(),
            MaxPool2d(kernel_size=2),
            Conv2d(32, 64, kernel_size=3, padding=1),
            ReLU(),
            MaxPool2d(kernel_size=2),
            AdaptiveAvgPool2d(1),
            Flatten(),
            Dropout(p=0.25),
            Linear(64, output_size),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)

In [ ]:
def predict(model, features, indices, criterion=None, targets=None, batch=256):
    """Previsão em lotes (não estoura a memória da GPU com o fold inteiro). Devolve (previsões, loss média)."""
    model.eval()
    preds = []
    soma_loss = 0.0
    with torch.no_grad():
        for start in range(0, len(indices), batch):
            idx = indices[start:start + batch]
            x = (torch.from_numpy(features[idx]).float() / escala).to(device)
            output = model(x)
            preds.append(torch.argmax(output, dim=1).cpu().numpy())
            if criterion is not None:
                y = torch.from_numpy(targets[idx]).long().to(device)
                soma_loss += criterion(output, y).item() * len(idx)
    model.train()
    return np.concatenate(preds), soma_loss / len(indices)


def class_weights(labels, num_classes):
    # Peso inverso à frequência: n_total / (n_classes * n_da_classe)
    counts = np.bincount(labels, minlength=num_classes)
    weights = len(labels) / (num_classes * np.maximum(counts, 1))
    return torch.tensor(weights, dtype=torch.float32, device=device)


def train_model_class(features, targets, groups, kf, batch, epochs, exp=False, lr=0.001):
    print(f"Treinando na GPU: {torch.cuda.get_device_name(0)}")
    acc_scores = []
    f1_scores = []
    bacc_scores = []
    folds_loss = []
    folds_acc = []
    folds_val = []
    folds_time = []
    cms = []
    num_classes = len(class_names)
    for i, (train_index, test_index) in enumerate(kf.split(features, targets, groups)):
        print(f"Treinando modelo ----- Fold {i + 1}")
        inicio = time.perf_counter()
        torch.manual_seed(42)

        # Validação: separa parte do treino (por lesão, se usar_grupos) para acompanhar a generalização por época
        val_index = np.array([], dtype=int)
        if validacao:
            val_split = StratifiedGroupKFold(n_splits=val_splits, shuffle=True, random_state=42) if usar_grupos \
                else StratifiedKFold(n_splits=val_splits, shuffle=True, random_state=42)
            tr_pos, val_pos = next(val_split.split(features[train_index], targets[train_index], groups[train_index]))
            val_index = train_index[val_pos]
            train_index = train_index[tr_pos]

        weights = class_weights(targets[train_index], num_classes) if peso_classes else None
        criterion = CrossEntropyLoss(weight=weights)

        train_data = MyDataset(features[train_index], targets[train_index])
        train_loader = DataLoader(train_data, batch_size=batch, shuffle=True)
        model = CustomModel(16900, num_classes).to(device)
        optmizer = Adam(model.parameters(), lr=lr)
        epoch_loss = []
        epoch_acc = []
        val_hist = {"epoca": [], "loss": [], "f1": []}
        for i_epochs in range(epochs):
            model.train()
            soma_loss = torch.zeros((), device=device)
            acertos = torch.zeros((), device=device)
            total = 0
            for x, y in train_loader:
                optmizer.zero_grad()
                output = model(x)
                loss = criterion(output, y)
                loss.backward()
                optmizer.step()
                soma_loss += loss.detach() * len(y)
                acertos += (output.argmax(dim=1) == y).sum()
                total += len(y)

            epoch_loss.append((soma_loss / total).item())
            epoch_acc.append((acertos / total).item())

            if validacao and ((i_epochs + 1) % val_cada == 0 or i_epochs + 1 == epochs):
                val_pred, val_loss = predict(model, features, val_index, criterion, targets)
                val_hist["epoca"].append(i_epochs + 1)
                val_hist["loss"].append(val_loss)
                val_hist["f1"].append(f1_score(targets[val_index], val_pred, average="macro"))

        del train_data, train_loader
        folds_time.append(time.perf_counter() - inicio)
        folds_loss.append(epoch_loss)
        folds_acc.append(epoch_acc)
        folds_val.append(val_hist)

        y_true = targets[test_index]
        result, _ = predict(model, features, test_index)
        acc_s = accuracy_score(y_true, result)
        f1_s = f1_score(y_true, result, average="macro")
        bacc_s = balanced_accuracy_score(y_true, result)
        cm = confusion_matrix(y_true, result, labels=range(num_classes))
        acc_scores.append(acc_s)
        f1_scores.append(f1_s)
        bacc_scores.append(bacc_s)
        cms.append(cm)
        print(f"Acc: {acc_s}")
        print(f"F1-Score (macro): {f1_s}")
        print(f"Acc balanceada: {bacc_s}")
        if validacao:
            best = int(np.argmax(val_hist["f1"]))
            print(f"Melhor época na validação: {val_hist['epoca'][best]} (F1 val: {val_hist['f1'][best]:.4f})")
        print(f"CM:\n{cm}")

        if exp:
            break

    print(f"\nResultado modelo ----- Acc: {np.mean(acc_scores)} ± {np.std(acc_scores)}, "
          f"F1-Score: {np.mean(f1_scores)} ± {np.std(f1_scores)}, Acc balanceada: {np.mean(bacc_scores)} ± {np.std(bacc_scores)}")

    return folds_loss, folds_acc, folds_val, folds_time, acc_scores, f1_scores, bacc_scores, cms

In [ ]:
def format_time(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))


def format_mean_std(scores):
    return f"{np.mean(scores) * 100:.2f}% ± {np.std(scores) * 100:.2f}%"


def plot_confusion_matrix(ax, cm, class_names, title):
    # Normalizada por linha (% de cada classe real): com classes desbalanceadas, a contagem esconde os erros nas classes pequenas
    cm_pct = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1) * 100
    disp = ConfusionMatrixDisplay(cm_pct, display_labels=class_names)
    disp.plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45, values_format=".0f")
    ax.set_xlabel("Previsto")
    ax.set_ylabel("Real (% da linha)")
    ax.set_title(title)


def plot_loss(folds_loss, folds_acc, folds_val, folds_time, acc_scores, f1_scores, bacc_scores, cms, class_names, dataset, dataset_info, max_epochs, batch, lr, n_folds, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    folds_acc = np.array(folds_acc) * 100
    for idx, fold_l in enumerate(folds_loss):
        fig, (ax1, ax2, ax_cm) = plt.subplots(1, 3, figsize=(20, 5.5), gridspec_kw={"width_ratios": [1.3, 1.3, 1]})
        x = np.arange(1, len(fold_l) + 1)
        val = folds_val[idx]
        ax1.plot(x, fold_l, color="red", label="Erro treino")
        if val["epoca"]:
            ax1.plot(val["epoca"], val["loss"], color="orange", label="Erro validação")
        ax1.set_xlabel("Épocas")
        ax1.set_ylabel("Erro")
        ax1.set_title("Erro")
        ax1.grid()
        ax1.legend()

        ax2.plot(x, folds_acc[idx], color="blue", label="Acc treino")
        if val["epoca"]:
            ax2.plot(val["epoca"], np.array(val["f1"]) * 100, color="green", label="F1 macro validação")
            best = int(np.argmax(val["f1"]))
            ax2.axvline(val["epoca"][best], color="green", linestyle="--", alpha=0.6, label=f"Melhor época val: {val['epoca'][best]}")
        ax2.set_xlabel("Épocas")
        ax2.set_ylabel("%")
        ax2.set_title("Acurácia (treino) e F1 macro (validação)")
        ax2.grid()
        ax2.legend()

        plot_confusion_matrix(ax_cm, cms[idx], class_names, "Matriz de confusão (teste)")

        fig.suptitle(
            f"{dataset} | Fold {idx + 1}/{n_folds} | GPU: {torch.cuda.get_device_name(0)}\n"
            f"{dataset_info}\n"
            f"Épocas: {max_epochs} | Batch: {batch} | LR: {lr} | Tempo: {format_time(folds_time[idx])}\n"
            f"Teste ----- Acc: {acc_scores[idx] * 100:.2f}% | F1-Score (macro): {f1_scores[idx] * 100:.2f}% | Acc balanceada: {bacc_scores[idx] * 100:.2f}%"
        )
        fig.tight_layout()
        fig.savefig(output_dir / f"fold_{idx + 1}.png", dpi=150, bbox_inches="tight")
        fig.show()


def plot_summary(folds_loss, folds_acc, folds_val, folds_time, acc_scores, f1_scores, bacc_scores, cms, class_names, dataset, dataset_info, max_epochs, batch, lr, n_folds, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    folds_acc = np.array(folds_acc) * 100
    fig, (ax1, ax2, ax_cm) = plt.subplots(1, 3, figsize=(20, 5.5), gridspec_kw={"width_ratios": [1.3, 1.3, 1]})
    x = np.arange(1, len(folds_loss[0]) + 1)
    for idx, fold_l in enumerate(folds_loss):
        ax1.plot(x, fold_l, alpha=0.4, label=f"Fold {idx + 1} treino ({format_time(folds_time[idx])})")
        ax2.plot(x, folds_acc[idx], alpha=0.4, label=f"Fold {idx + 1} (F1 teste: {f1_scores[idx] * 100:.2f}%)")

    ax1.plot(x, np.mean(folds_loss, axis=0), color="black", linewidth=2, label="Média treino")
    ax2.plot(x, folds_acc.mean(axis=0), color="black", linewidth=2, label="Média acc treino")

    # Média da validação entre os folds
    if folds_val[0]["epoca"]:
        ep = folds_val[0]["epoca"]
        ax1.plot(ep, np.mean([v["loss"] for v in folds_val], axis=0), color="orange", linewidth=2, label="Média validação")
        val_f1 = np.mean([v["f1"] for v in folds_val], axis=0) * 100
        ax2.plot(ep, val_f1, color="green", linewidth=2, label="Média F1 macro validação")
        best = int(np.argmax(val_f1))
        ax2.axvline(ep[best], color="green", linestyle="--", alpha=0.6, label=f"Melhor época (média val): {ep[best]}")

    ax1.set_xlabel("Épocas")
    ax1.set_ylabel("Erro")
    ax1.set_title("Erro no treino e na validação")
    ax1.grid()
    ax1.legend(fontsize=8)
    ax2.set_xlabel("Épocas")
    ax2.set_ylabel("%")
    ax2.set_title("Acc no treino e F1 macro na validação (legenda: F1 no teste)")
    ax2.grid()
    ax2.legend(fontsize=8)

    # Soma das matrizes de todos os folds: cada imagem aparece uma vez no teste
    plot_confusion_matrix(ax_cm, np.sum(cms, axis=0), class_names, "Matriz de confusão (soma dos folds)")

    fig.suptitle(
        f"{dataset} | Folds: {len(folds_loss)}/{n_folds} | GPU: {torch.cuda.get_device_name(0)}\n"
        f"{dataset_info}\n"
        f"Épocas: {max_epochs} | Batch: {batch} | LR: {lr} | Tempo total: {format_time(sum(folds_time))}\n"
        f"Teste ----- Acc: {format_mean_std(acc_scores)} | F1-Score (macro): {format_mean_std(f1_scores)} | Acc balanceada: {format_mean_std(bacc_scores)}"
    )
    fig.tight_layout()
    fig.savefig(output_dir / "resumo.png", dpi=150, bbox_inches="tight")
    print(f"Gráficos salvos em: {output_dir}")
    fig.show()


def save_results(results_csv, acc_scores, f1_scores, folds_time, dataset, images_per_class, image_size, color_mode, normalizar, max_epochs, batch, lr, n_folds, output_dir):
    # Mesmas colunas do resultados.csv dos outros notebooks (grupos, pesos e validação vão no nome do dataset)
    # Separador ";" e vírgula decimal para abrir direto no Excel em português
    def num(value):
        return f"{value:.2f}".replace(".", ",")

    row = {
        "data_hora": time.strftime("%Y-%m-%d %H:%M:%S"),
        "dataset": dataset,
        "modo": color_mode,
        "normalizado": "Sim" if normalizar else "Não",
        "imagens_total": sum(images_per_class.values()),
        "imagens_por_classe": " | ".join(f"{name}: {count}" for name, count in images_per_class.items()),
        "tamanho": image_size,
        "epocas": max_epochs,
        "batch": batch,
        "lr": str(lr).replace(".", ","),
        "folds": f"{len(acc_scores)}/{n_folds}",
        "gpu": torch.cuda.get_device_name(0),
        "tempo_total": format_time(sum(folds_time)),
        "acc_media_%": num(np.mean(acc_scores) * 100),
        "acc_desvio_%": num(np.std(acc_scores) * 100),
        "f1_media_%": num(np.mean(f1_scores) * 100),
        "f1_desvio_%": num(np.std(f1_scores) * 100),
        "acc_por_fold_%": " | ".join(num(acc * 100) for acc in acc_scores),
        "f1_por_fold_%": " | ".join(num(f1 * 100) for f1 in f1_scores),
        "pasta_graficos": str(output_dir),
    }

    results_csv.parent.mkdir(parents=True, exist_ok=True)
    new_file = not results_csv.exists()
    with open(results_csv, "a", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=list(row), delimiter=";")
        if new_file:
            writer.writeheader()
        writer.writerow(row)
    print(f"Resultado adicionado em: {results_csv}")

In [ ]:
def load_data(data_dir: Path, class_names, limit=None, size=(128, 128), seed=42):
    """Lê o HAM10000: rótulo (dx) e lesão (lesion_id) vêm do HAM10000_metadata.csv; as imagens ficam em part_1 e part_2.
    limit: máximo de imagens por classe (sorteadas com seed fixa); None usa todas."""
    meta = pd.read_csv(data_dir / "HAM10000_metadata.csv")

    # Mapeia image_id -> arquivo nas duas pastas
    paths = {}
    for folder in ["HAM10000_images_part_1", "HAM10000_images_part_2"]:
        for p in (data_dir / folder).iterdir():
            if p.is_file():
                paths[p.stem] = p

    if limit is not None:
        meta = meta.sample(frac=1, random_state=seed).groupby("dx").head(limit)
    meta = meta.sort_values("image_id").reset_index(drop=True)

    data_array, labels, groups = [], [], []
    for row in meta.itertuples():
        image_path = paths.get(row.image_id)
        if image_path is None:
            print(f"Imagem não encontrada: {row.image_id}")
            continue
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)

        if image is None:
            print(f"Não foi possível carregar: {image_path.name}")
            continue
        # O OpenCV lê em BGR; converte para RGB
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, size)
        data_array.append(image)
        labels.append(class_names.index(row.dx))
        groups.append(row.lesion_id)

    return np.asarray(data_array, dtype=np.uint8), np.asarray(labels, dtype=int), np.asarray(groups)

In [ ]:
# Pasta com HAM10000_metadata.csv, HAM10000_images_part_1 e HAM10000_images_part_2
# Docker
# DATA_DIR = Path("/app/data/Skin_Cancer")
# VS Code (PC)
DATA_DIR = Path("../data/Skin_Cancer")

# Nomes das classes na ordem dos targets (0, 1, 2...) — siglas da coluna dx do metadata
# akiec: queratose actínica | bcc: carcinoma basocelular | bkl: lesão queratósica benigna | df: dermatofibroma
# mel: melanoma | nv: nevo melanocítico | vasc: lesão vascular
class_names = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]

# limit=None usa as 10.015 imagens; limit=1000 limita cada classe a 1000 (só a nv e as de >1000 são cortadas)
limit = None

features, targets, groups = load_data(DATA_DIR, class_names, limit=limit)
features = features.transpose(0, 3, 1, 2)  # (N, 128, 128, 3) -> (N, 3, 128, 128)

In [ ]:
print(features.shape)
print(dict(zip(class_names, np.bincount(targets).tolist())))
print(f"Lesões distintas: {len(np.unique(groups))} | Imagens: {len(groups)}")

In [ ]:
dataset = "Pele cor"
max_epochs = 600
batch = 64
lr = 0.001
normalizar = True
escala = 255 if normalizar else 1

# Divisão por lesão: imagens da mesma lesão (lesion_id) ficam todas no treino ou todas no teste (evita vazamento)
usar_grupos = True
# Peso por classe na loss (inverso da frequência): sem isso o modelo tende a prever quase tudo como nv
peso_classes = True
# Validação: 1/val_splits do treino de cada fold vira validação, avaliada a cada val_cada épocas
validacao = True
val_splits = 7
val_cada = 5

kf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42) if usar_grupos \
    else StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Configuração vai no nome do dataset (aparece nos gráficos, na pasta e no resultados.csv)
dataset += (" grupos" if usar_grupos else " sem grupos") + (" pesos" if peso_classes else " sem pesos")
if limit is not None:
    dataset += f" limit{limit}"

# Informações do dataset (aparecem nos gráficos e no resultados.csv)
images_per_class = dict(zip(class_names, np.bincount(targets, minlength=len(class_names)).tolist()))
image_size = f"{features.shape[3]}x{features.shape[2]}"
color_mode = "RGB" if features.shape[1] == 3 else "Cinza"
class_counts = " | ".join(f"{name}: {count}" for name, count in images_per_class.items())
dataset_info = (f"Imagens: {len(targets)} ({class_counts}) | Tamanho: {image_size} | {color_mode} | "
                f"Normalizado: {'Sim' if normalizar else 'Não'}\n"
                f"Divisão por lesão: {'Sim' if usar_grupos else 'Não'} | Peso por classe: {'Sim' if peso_classes else 'Não'} | "
                f"Validação: {'Sim (1/' + str(val_splits) + ' do treino, a cada ' + str(val_cada) + ' épocas)' if validacao else 'Não'}")

# Pasta onde os gráficos são salvos (uma subpasta por execução, para não sobrescrever)
# Docker
# output_dir = Path("/app/graficos") / (dataset.replace(" ", "_") + ("_norm" if normalizar else "")) / time.strftime("%Y-%m-%d_%H-%M-%S")
# VS Code (PC)
output_dir = Path("../graficos") / (dataset.replace(" ", "_") + ("_norm" if normalizar else "")) / time.strftime("%Y-%m-%d_%H-%M-%S")

# Tabela com uma linha por execução, compartilhada por todos os notebooks
# Docker
# results_csv = Path("/app/graficos/resultados.csv")
# VS Code (PC)
results_csv = Path("../graficos/resultados.csv")

In [ ]:
folds_loss, folds_acc, folds_val, folds_time, acc_scores, f1_scores, bacc_scores, cms = train_model_class(features, targets, groups, kf, batch, max_epochs, False, lr)

In [ ]:
plot_loss(folds_loss, folds_acc, folds_val, folds_time, acc_scores, f1_scores, bacc_scores, cms, class_names, dataset, dataset_info, max_epochs, batch, lr, kf.get_n_splits(), output_dir)

In [ ]:
plot_summary(folds_loss, folds_acc, folds_val, folds_time, acc_scores, f1_scores, bacc_scores, cms, class_names, dataset, dataset_info, max_epochs, batch, lr, kf.get_n_splits(), output_dir)

In [ ]:
save_results(results_csv, acc_scores, f1_scores, folds_time, dataset, images_per_class, image_size, color_mode, normalizar, max_epochs, batch, lr, kf.get_n_splits(), output_dir)